# {ANALYSIS_TITLE}

**Analysis type:** {ANALYSIS_TYPE}  
**Target accession:** set `ACCESSION` in Section 2  
**Created:** {DATE}  

---

## Sections
1. Environment setup
2. Accession & configuration
3. AFDB API data fetch
4. Structure parsing
5. Metric computation
6. Visualisation
7. Flywheel result submission *(optional)*

**Dependencies:** `numpy`, `matplotlib`, `seaborn`, `requests`, `molviewspec`, `ipywidgets`  
**Prohibited:** `biopython`, `torch`, `torch-geometric`

**This scaffold sits on `src/insightfold/complex_interface_utils.py`.** Fetching, parsing, interface detection, the seven confidence scores and the thresholds all come from there, so a new analysis type starts from verified code rather than from a copy. The authorities are `specs/homodimer_diagnostic/formula-reference.md` (formulas) and `specs/homodimer_diagnostic/threshold-reference.md` (thresholds); see `CLAUDE.md` for the map.


---
## 1. Environment Setup

Installs the allowed third-party packages when they are missing, then puts the repo's
`src/` on `sys.path` so `insightfold.complex_interface_utils` is importable.

**InsightFold is deliberately not `pip install`ed** (decision D9): that would resolve
`pyproject.toml` and pull in `biopython`, `gemmi`, `scipy` and `plotly`, breaking both the
project's dependency rules and the 60 s Colab install budget.


In [ ]:
# Bootstrap. This one cell works unchanged locally and on Google Colab.
#
# Third-party packages are installed only when missing. InsightFold itself is
# deliberately NOT pip-installed: that would resolve pyproject.toml and drag in
# biopython, gemmi, scipy and plotly, breaking both the project's dependency
# rules and the 60 s Colab install budget (decision D9 in
# specs/homodimer_diagnostic/rework-plan.md). The repo is cloned shallowly on
# Colab instead, and src/ is put on sys.path.
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO_URL = 'https://github.com/PDBeurope/InsightFold.git'
# TODO(merge): flip REPO_BRANCH to 'main' (or a release tag) once the
# homodimer-notebook-rework branch merges. complex_interface_utils exists only
# on that branch today. Grep for "TODO(merge)" before releasing this notebook.
REPO_BRANCH = 'homodimer-notebook-rework'
COLAB_CLONE_DIR = Path('/content/InsightFold')

PACKAGES = ['numpy', 'matplotlib', 'seaborn', 'requests', 'molviewspec', 'ipywidgets']
_missing = [p for p in PACKAGES if find_spec(p) is None]
if _missing:
    print(f'Installing: {", ".join(_missing)}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start=None):
    """Walk up from `start` (default: cwd) for a dir holding pyproject.toml and src/."""
    s = (Path(start) if start is not None else Path.cwd()).expanduser().resolve()
    for d in (s, *s.parents):
        if (d / 'pyproject.toml').is_file() and (d / 'src').is_dir():
            return d
    return None


REPO_ROOT = find_repo_root()
if IN_COLAB and REPO_ROOT is None:
    print(f'Cloning {REPO_URL} ({REPO_BRANCH}) ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                           REPO_URL, str(COLAB_CLONE_DIR)])
    REPO_ROOT = find_repo_root(COLAB_CLONE_DIR)

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the InsightFold checkout.\n'
        f'Looked upwards from {Path.cwd()} for a directory containing both '
        "'pyproject.toml' and 'src/'. Run this notebook from inside the repo, "
        'or set REPO_ROOT by hand.')

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Environment: {"Colab" if IN_COLAB else "local"}')
print(f'Repo root  : {REPO_ROOT}')


In [ ]:
import base64
import json

import matplotlib.pyplot as plt
import numpy as np
import requests
import seaborn as sns
from IPython.display import HTML, IFrame, display

# One module backs every analysis in this repo (decision D1/D8: named by domain,
# not by notebook). Fetching, parsing, interface detection, the seven scores,
# the thresholds and the plots all live here, verified against ipsae.py v4.
from insightfold import complex_interface_utils as ciu

sns.set_style('white')
# No blanket warnings.filterwarnings('ignore'): it hides the deprecations you
# most need to see. Suppress a specific warning if one actually gets in the way.
print(f'Imports OK. Module loaded from {ciu.__file__}')


---
## 2. Accession & Configuration

Set `ACCESSION` to an AlphaFold DB accession (e.g. `AF-0000000065889468`)  
or set `USE_LOCAL_FILE = True` and upload your own mmCIF/PAE/pLDDT files.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
ACCESSION     = 'AF-0000000065889468'   # AFDB accession for a homodimer
USE_LOCAL_FILE = False                   # True → upload widgets appear below
DIST_CUTOFF    = 8.0                     # CB–CB contact cutoff in Å
# ──────────────────────────────────────────────────────────────────────────────

AFDB_META_URL = 'https://alphafold.ebi.ac.uk/api/prediction/{acc}'
print(f'Accession: {ACCESSION}')

In [ ]:
# Local file upload (only shown when USE_LOCAL_FILE = True)
if USE_LOCAL_FILE:
    import ipywidgets as widgets
    upload_cif    = widgets.FileUpload(description='mmCIF',   multiple=False)
    upload_pae    = widgets.FileUpload(description='PAE JSON', multiple=False)
    upload_plddt  = widgets.FileUpload(description='pLDDT JSON', multiple=False)
    display(upload_cif, upload_pae, upload_plddt)
    print('Upload files then run the next cell.')
else:
    print('Online mode: data will be fetched from AFDB REST API.')

---
## 3. AFDB API Data Fetch

Fetches metadata from `https://alphafold.ebi.ac.uk/api/prediction/{accession}`,
then downloads mmCIF, PAE JSON, and pLDDT JSON via the URLs in that response.

In [ ]:
if USE_LOCAL_FILE:
    # Read from the upload widgets (run after uploading in Section 2).
    prediction = None
    cif_text  = list(upload_cif.value.values())[0]['content'].decode()
    pae_raw   = json.loads(list(upload_pae.value.values())[0]['content'].decode()) if upload_pae.value else None
    plddt_raw = json.loads(list(upload_plddt.value.values())[0]['content'].decode()) if upload_plddt.value else None
    struct_url, struct_fmt = None, None
    print('Loaded from local files.')
else:
    prediction = ciu.fetch_afdb_metadata(ACCESSION)

    # The prediction endpoint returns ONE ENTRY PER CHAIN, in NON-DETERMINISTIC
    # order: the same accession answers ['A', 'B'] on one call and ['B', 'A'] on
    # the next. `entries[0]` is therefore not merely wrong but non-deterministically
    # wrong, and on a heterodimer it flips the reported identity between chains
    # from run to run. `AFDBPrediction` sorts entries by chain id at construction
    # and every field below is read by chain id, never by position.
    print(f'Chains described: {list(prediction.chain_ids)}')

    print('Downloading mmCIF, PAE JSON and pLDDT JSON ...')
    cif_text  = ciu.download_structure(prediction)
    pae_raw   = ciu.download_pae(prediction)
    plddt_raw = ciu.download_plddt(prediction)

    # One decision for URL and format together: a present-but-empty bcifUrl
    # labelled 'bcif' is a silently blank 3D viewer.
    _source = ciu.resolve_structure_source(prediction)
    struct_url, struct_fmt = _source.url, _source.format
    print('All downloads complete.')

# --- Metadata, per chain ---
# Field names verified live: the service sends `gene` and `latestVersion`.
# It does NOT send `geneNames` or `modelVersion`; asking for those prints N/A
# on every run, which is what this cell used to do.
if prediction is not None:
    for _cid in prediction.chain_ids:
        _entry = prediction.entry_for_chain(_cid)
        print(f'\nChain {_cid}')
        print(f"  Protein : {_entry.get('uniprotDescription', 'N/A')}")
        print(f"  UniProt : {_entry.get('uniprotAccession', 'N/A')}")
        print(f"  Organism: {_entry.get('organismScientificName', 'N/A')}")
        print(f"  Gene    : {_entry.get('gene', 'N/A')}")
        print(f"  Version : {_entry.get('latestVersion', 'N/A')}")
        print(f"  Length  : {len(_entry.get('sequence') or '')} aa (monomer)")
    _first = prediction.entry_for_chain(prediction.chain_ids[0])
    print(f"\nAssembly: {_first.get('assemblyType', '?')} "
          f"{_first.get('oligomericState', '?')}  "
          f"(isComplex={_first.get('isComplex', '?')})")


---
## 4. Structure Parsing

Parses the mmCIF into per-chain CB/CA coordinate arrays via `ciu.parse_structure`, then
resolves the PAE and pLDDT documents against it. Chain ids are reconciled across all three
sources and **never paired positionally**: a mismatch fails loudly rather than mis-slicing
every PAE quadrant into plausible but wrong scores.


In [ ]:
# mmCIF parsing lives in the module. The repo used to carry three divergent
# inline copies of this parser, which disagreed on whether the `_atom_site.`
# prefix was stripped; R015 folded them into one and R095 deleted the last copy.
# The module's version also fixes a latent bug the old copies shared: a blank
# line inside the _atom_site loop is NOT a terminator, and treating it as one
# silently dropped every atom after it.
chains = ciu.parse_structure(cif_text)

ch_coords   = {cid: c.coords    for cid, c in chains.items()}
ch_resids   = {cid: c.res_ids   for cid, c in chains.items()}
ch_resnames = {cid: c.res_names for cid, c in chains.items()}
ch_plddt    = {cid: c.plddt     for cid, c in chains.items()}
chain_ids   = sorted(chains)

print(f'Chains found: {chain_ids}')
for _ch in chain_ids:
    print(f'  Chain {_ch}: {chains[_ch].n_residues} residues')


In [ ]:
# ── PAE + pLDDT ARRAYS ────────────────────────────────────────────────────────

_structure_lengths = {cid: c.n_residues for cid, c in chains.items()}
pae   = ciu.parse_pae(pae_raw, fallback_lengths=_structure_lengths)
plddt = ciu.parse_plddt(plddt_raw, fallback_lengths=_structure_lengths)

# Three sources describe the chains (mmCIF, PAE, pLDDT) and every quadrant slice
# below assumes all three agree. A disagreement misaligns the slices and yields
# plausible but wrong scores rather than an error, so agreement is a checked
# precondition. Chain ids are never paired positionally. This is also the dimer
# gate: a monomer or a three-chain model is refused here with an explanation.
identity = ciu.verify_chain_identity(chains, pae, plddt, prediction)
print(identity.assembly.headline)
for _note in identity.notes:
    print(f'NOTE: {_note}')

# D4: everything downstream takes one explicit ORDERED chain pair. PAE is
# asymmetric (pae_AB != pae_BA.T), so the order is part of the query.
chain_x, chain_y = pae.chain_ids[0], pae.chain_ids[1]
pair    = pae.ordered_pair(chain_x, chain_y)
nA, nB  = pair.nx, pair.ny

pae_matrix = pae.matrix
pae_max    = pae.max_pae
pae_AA, pae_AB = pair.block_xx, pair.block_xy
pae_BA, pae_BB = pair.block_yx, pair.block_yy

plddt_A = plddt.for_chain(chain_x)
plddt_B = plddt.for_chain(chain_y)

print(f'\nPAE matrix: {pae_matrix.shape}   nA={nA} ({chain_x})   nB={nB} ({chain_y})')
print(f'pLDDT mean: {chain_x}={plddt_A.mean():.1f}  {chain_y}={plddt_B.mean():.1f}')


---
## 5. Metric Computation

All seven confidence values come from `insightfold.complex_interface_utils`, which is the
executable transcription of `specs/homodimer_diagnostic/formula-reference.md` (line-cited to
`ipsae.py` v4) and `specs/homodimer_diagnostic/threshold-reference.md`.

**Do not reimplement a score in this notebook.** Four of the seven are easy to get subtly
wrong, and every inline copy that has ever existed in this repo was wrong on at least one of
them. Each `compute_*` returns a result object carrying the final score *and* the
intermediates (masks, counts, `d0` values, per-residue profiles), so nothing needs
recomputing for the visualisations below.


In [ ]:
# ── PRIMITIVE FUNCTIONS ───────────────────────────────────────────────────────
# ipsae.py has TWO d0 helpers and they are not interchangeable:
#   ciu.d0_scalar(L)  -> d0chn (ipTM_d0chn, ipSAE_d0chn) and d0dom
#   ciu.d0_array(L)   -> d0res ONLY, per residue
# They are bit-exact for every integer L except L == 27. See
# specs/homodimer_diagnostic/formula-reference.md, which line-cites both.

d0_scalar, d0_array, ptm_func = ciu.d0_scalar, ciu.d0_array, ciu.ptm_func

print(f'd0_scalar(27) = {d0_scalar(27):.6f}')
print(f'd0_array(27)  = {float(d0_array(27)):.6f}   <- 19x the +-0.001 tolerance apart')


In [ ]:
# ── INTERFACE DETECTION ───────────────────────────────────────────────────────
# CB-CB distance <= DIST_CUTOFF, with CA substituted for glycine, matching the
# IPSAE scoring code's contact definition. (AFDB's production pipeline uses
# CA-CA; do not "fix" this without changing the scoring functions too.)

contacts = ciu.detect_interface(chains[chain_x], chains[chain_y],
                                dist_cutoff=DIST_CUTOFF)

coords_A, coords_B = chains[chain_x].coords, chains[chain_y].coords
dist_matrix = contacts.dist_matrix          # (nA, nB)
contact_mat = contacts.contact_mask
if_A, if_B  = contacts.mask_x, contacts.mask_y

# Two different numbers that must never be conflated. pDockQ needs the PAIR
# count; on a typical interface it is several times the residue count.
n_contact_pairs      = contacts.n_contact_pairs
n_interface_residues = contacts.n_interface_residues

print(f'Contact cutoff        : {DIST_CUTOFF} A (CB-CB, CA for GLY)')
print(f'Contact PAIRS         : {n_contact_pairs}')
print(f'Interface RESIDUES    : {n_interface_residues} '
      f'({chain_x}: {int(if_A.sum())}, {chain_y}: {int(if_B.sum())})')


In [ ]:
# ── SCORE COMPUTATION ─────────────────────────────────────────────────────────
# Extend this cell for non-dimer analysis types, but do NOT reimplement the
# scores. Every formula here is the module's, transcribed and line-cited in
# specs/homodimer_diagnostic/formula-reference.md and verified against
# ipsae.py v4 within +-0.001 on ten fixtures. The inline copies this cell used
# to hold were wrong on pDockQ (residue count instead of pair count), pDockQ2
# (wrong sigmoid constants and wrong pLDDT aggregation), LIS (max instead of
# mean) and d0dom (not recomputed per direction).

res_iptm   = ciu.compute_iptm_d0chn(pair)
res_ipsae  = ciu.compute_ipsae(pair, pae_cutoff=ciu.PAE_CUTOFF)
res_pdockq = ciu.compute_pdockq(contacts, plddt_A, plddt_B)
res_pdockq2 = ciu.compute_pdockq2(contacts, pair, plddt_A, plddt_B)
res_lis    = ciu.compute_lis(pair)

scores = {
    'iptm_d0chn':  res_iptm.score,
    'ipsae_d0res': res_ipsae.variants['ipsae_d0res'].score,
    'ipsae_d0chn': res_ipsae.variants['ipsae_d0chn'].score,
    'ipsae_d0dom': res_ipsae.variants['ipsae_d0dom'].score,
    'pdockq':      res_pdockq.score,
    'pdockq2':     res_pdockq2.score,
    'lis':         res_lis.score,
}

print('Confidence scores (each the ipsae.py "max" row):')
for _k, _v in scores.items():
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_k]:14s} {_v:.4f}')


In [ ]:
# ── TRAFFIC-LIGHT CLASSIFICATION ──────────────────────────────────────────────
# Thresholds come from the module, which transcribes
# specs/homodimer_diagnostic/threshold-reference.md. Never hard-code a copy:
# the dict that used to live in this cell was wrong on four of its five values,
# and amber must be read from the table, never recomputed as green / 2.

summary = ciu.summarise_scores(scores)
verdict = ciu.afdb_high_confidence(scores['ipsae_d0res'], scores['pdockq2'])

COLOUR_MAP = {'green': '\033[92m', 'amber': '\033[93m', 'red': '\033[91m'}
RESET = '\033[0m'

print('Classification:')
for _k, _v in scores.items():
    _colour, _label = ciu.traffic_light(_v, _k)
    _t = ciu.THRESHOLDS[_k]
    print(f'  {COLOUR_MAP[_colour]}{ciu.SCORE_DISPLAY_NAMES[_k]:14s} {_v:.4f}  '
          f'[{_label}]{RESET}  green >= {_t.green}, amber >= {_t.amber}  ({_t.green_provenance})')

# The headline verdict is AFDB's joint criterion, not any single traffic light.
# The seven lights are diagnostics that explain it, not seven independent votes,
# and three of them (the ipSAE variants) are one measurement seen three ways.
print(f'\nAFDB high-confidence criterion: {verdict.verdict}')
print(f'  {verdict.reason}')
print(f'\n{summary.overall}')


---
## 6. Visualisation

In [ ]:
# ── 2D: PAE HEATMAP ───────────────────────────────────────────────────────────

if pae_matrix is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(pae_matrix, cmap='Greens_r', vmin=0, vmax=pae_max, origin='upper')
    ax.axhline(nA - 0.5, color='white', linewidth=1.2)
    ax.axvline(nA - 0.5, color='white', linewidth=1.2)
    ax.set_xlabel('Residue index')
    ax.set_ylabel('Residue index')
    ax.set_title(f'Predicted Aligned Error — {ACCESSION}')
    plt.colorbar(im, ax=ax, label='PAE (Å)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 2D: pLDDT PROFILE ─────────────────────────────────────────────────────────

if plddt_raw:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(range(1, nA + 1), plddt_A, color='#009688', label=f'Chain A (n={nA})')
    ax.plot(range(nA + 1, nA + nB + 1), plddt_B, color='#E91E63', label=f'Chain B (n={nB})')
    # Shade interface residues
    for i, is_if in enumerate(if_A):
        if is_if:
            ax.axvspan(i + 0.5, i + 1.5, color='#009688', alpha=0.15)
    for j, is_if in enumerate(if_B):
        if is_if:
            ax.axvspan(nA + j + 0.5, nA + j + 1.5, color='#E91E63', alpha=0.15)
    ax.axhline(70, color='gray', linestyle='--', linewidth=0.8, label='pLDDT=70')
    ax.set_xlabel('Residue index')
    ax.set_ylabel('pLDDT')
    ax.set_title('Per-residue pLDDT (shaded = interface)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 3D: MOLVIEWSPEC VIEWER ────────────────────────────────────────────────────
# Wrapped in try/except — degrades gracefully if molviewspec is not installed.

try:
    import molviewspec as mvs

    COLOUR_A = '#009688'
    COLOUR_B = '#E91E63'
    COLOUR_IF = '#FFD600'

    def _build_struct(url, fmt):
        builder = mvs.create_builder()
        return builder, (
            builder
            .download(url=url)
            .parse(format=fmt)
            .model_structure()
        )

    def show_mol_view(state, label, width=950, height=550):
        encoded = base64.b64encode(state.molstar_html().encode()).decode()
        display(HTML(f'<h4 style="font-family:sans-serif">{label}</h4>'))
        display(IFrame(src=f'data:text/html;base64,{encoded}', width=width, height=height))

    if struct_url:
        # View 1: cartoon coloured by chain
        builder, structure = _build_struct(struct_url, struct_fmt)
        for chain_label, colour in [(chain_ids[0], COLOUR_A), (chain_ids[1], COLOUR_B)]:
            (structure
             .component(selector=mvs.ComponentExpression(label_asym_id=chain_label))
             .representation(type='cartoon')
             .color(color=colour))
        show_mol_view(builder.get_state(), 'Chain colouring (A = teal, B = pink)')

        # View 2: interface highlighted
        builder2, structure2 = _build_struct(struct_url, struct_fmt)
        for chain_label, colour in [(chain_ids[0], COLOUR_A), (chain_ids[1], COLOUR_B)]:
            (structure2
             .component(selector=mvs.ComponentExpression(label_asym_id=chain_label))
             .representation(type='cartoon')
             .color(color=colour))
        # Colour interface residues yellow
        res_ids_A = ch_resids[chain_ids[0]]
        res_ids_B = ch_resids[chain_ids[1]]
        for idx, is_if in enumerate(if_A):
            if is_if:
                rid = int(res_ids_A[idx])
                (structure2
                 .component(selector=mvs.ComponentExpression(
                     label_asym_id=chain_ids[0], beg_label_seq_id=rid, end_label_seq_id=rid))
                 .representation(type='cartoon')
                 .color(color=COLOUR_IF))
        for idx, is_if in enumerate(if_B):
            if is_if:
                rid = int(res_ids_B[idx])
                (structure2
                 .component(selector=mvs.ComponentExpression(
                     label_asym_id=chain_ids[1], beg_label_seq_id=rid, end_label_seq_id=rid))
                 .representation(type='cartoon')
                 .color(color=COLOUR_IF))
        show_mol_view(builder2.get_state(), 'Interface residues highlighted (yellow)')
    else:
        print('No structure URL available (local-file mode) — skipping 3D views.\n      Nothing numerical is affected: every score above is already computed.')

except ImportError:
    print('molviewspec not installed — skipping 3D views.')

---
## 7. Flywheel Result Submission *(optional)*

Uncomment and configure this cell to write results to a downstream sink
(database, CSV file, or API endpoint). Remove if not applicable to this
analysis type.

In [ ]:
# ── FLYWHEEL RESULT SUBMISSION ────────────────────────────────────────────────
# Configure FLYWHEEL_ENDPOINT and FLYWHEEL_API_KEY as environment variables
# or Colab Secrets before enabling.

FLYWHEEL_ENABLED = False   # ← set True to activate

if FLYWHEEL_ENABLED:
    import os

    result_payload = {
        'accession':  ACCESSION,
        'analysis':   '{ANALYSIS_TYPE}',
        'scores':     scores,
        'n_contacts': n_contacts,
        'nA':         nA,
        'nB':         nB,
    }

    endpoint = os.environ.get('FLYWHEEL_ENDPOINT', '')
    api_key  = os.environ.get('FLYWHEEL_API_KEY', '')

    if not endpoint:
        print('FLYWHEEL_ENDPOINT not set — skipping submission.')
    else:
        resp = requests.post(
            endpoint,
            json=result_payload,
            headers={'Authorization': f'Bearer {api_key}'},
            timeout=15,
        )
        resp.raise_for_status()
        print(f'Result submitted: HTTP {resp.status_code}')
else:
    print('Flywheel submission disabled. Set FLYWHEEL_ENABLED = True to activate.')